# Demo 2 · Query NHL data with PandasAI + local Qwen

An **LLM** generates text or code from a prompt. **PandasAI** connects that model to a pandas table: it builds an analysis prompt, obtains code, and runs it to answer our question.

We use **PandasAI 2.3.2**, its `SmartDataframe`, and a small adapter for **Qwen2.5-1.5B-Instruct** running through Transformers.

**Natural-language question → Qwen generates Python → PandasAI executes it → inspect and verify**

PandasAI can return a number, a table, or a plot. It can also generate a plausible but wrong computation. We will compare three tasks with ordinary pandas using **MTL–CAR, May 21, 2026 (`2025030311`)**.

This notebook is self-contained. [Part 4](04_llm_and_rag.ipynb) retrieves *API documentation*; here we query *a loaded table*. PandasAI does not automatically use Part 4's documentation index.

## Setup: choose local or Colab

**Local development (see the [hardware recommendation](../../README_en.md#recommended-hardware-for-notebooks-04-and-05)):** run `uv sync --group llm` from the repository root and select the shared Python 3.11 `.venv` kernel. **Skip the optional installation cell** and continue with the environment check below. You do not need to install packages individually.

**Google Colab | no repository clone or manual uv installation needed:**

1. Open this notebook in Colab. Under **Runtime → Change runtime type**, choose **T4 GPU** and the **2025.07 runtime (Python 3.11)** if available. The default runtime may use a newer Python version; the installation cell cannot change the running interpreter. If no Python 3.11 runtime is available, use the local environment. [Colab runtime versions](https://research.google.com/colaboratory/runtime-version-faq.html)
2. In the **optional cell below**, set `INSTALL_COLAB_PACKAGES = True` and run it once. The cell installs `uv`, then uses it to install the libraries into the notebook's current Python. `sys.executable` is that Python's path; `subprocess.check_call` runs a command and stops if it fails. Colab already supplies PyTorch; the cell installs the additional dependencies.
3. Choose **Runtime → Restart session** after installation, even if no restart warning appears. This is needed because NumPy/pandas may already be loaded with different versions.
4. Set the flag back to `False`, then run the **environment check** and continue with **Load Qwen**. Repeat installation when Colab gives you a new runtime; a normal session restart keeps installed packages.

The first model download needs internet; inference then runs in your own runtime, without an API key. CPU mode works but is slower. Use a fresh Colab runtime containing only public NHL files when executing generated code: do not mount Drive or add credentials. For local execution of generated code, use a disposable environment with no access to personal files; a Python virtual environment alone is not a sandbox.

In [ ]:
# OPTIONAL: run once in a fresh Colab runtime; skip during local development.
INSTALL_COLAB_PACKAGES = False  # Set to True to install; reset to False afterward.

import sys
import subprocess

# Detect Colab; otherwise use the local .venv.
try:
    from google.colab import output
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if not IN_COLAB:
    print("Local environment: skipped. Use uv sync --group llm in your terminal.")
elif not INSTALL_COLAB_PACKAGES:
    print("Installation skipped. Set INSTALL_COLAB_PACKAGES = True if this is a fresh Colab runtime.")
else:
    if sys.version_info[:2] != (3, 11):
        raise RuntimeError("Choose a Python 3.11 Colab runtime before installing these packages.")
    # Install into the Python running these cells.
    subprocess.check_call([sys.executable, "-m", "pip", "install", "uv>=0.8,<1"])
    packages = [
        "ipython>=8,<9",
        "matplotlib-inline<0.2",
        "numpy==1.26.4",
        "pandas==1.5.3",
        "transformers==4.57.6",
        "accelerate>=1,<2",
        "bitsandbytes>=0.45,<1",
        "requests",
        "pandasai==2.3.2",
    ]
    subprocess.check_call([sys.executable, "-m", "uv", "pip", "install",
                           "--python", sys.executable, *packages])
    print("Installation finished. Choose Runtime → Restart session before continuing.")


### After installation: check the environment

On Colab, **restart the session first** and skip the installation cell on your next run. Locally, run this check after `uv sync --group llm`. Importing NumPy and pandas here also checks that their installed binaries load together.

In [ ]:
import sys

if sys.version_info[:2] != (3, 11):
    raise RuntimeError("Use a Python 3.11 kernel: the course .venv locally, or a compatible Colab runtime.")

import numpy as np
import pandas as pd
import torch
from importlib.metadata import version

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__, "| pandas:", pd.__version__)
print("Transformers:", version("transformers"))
print("PandasAI:", version("pandasai"))
print("CUDA GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Unavailable — CPU/Apple GPU fallback")

### Load Qwen

We use Hugging Face's pipeline and Qwen chat template. **4-bit** loading stores compressed weights on a CUDA GPU; CPU and Apple GPU runs use ordinary weights. `do_sample=False` uses greedy decoding; outputs can still differ across devices and package versions. The setup code is supplied—focus on the prompts and their results.

**What are we importing?**

- **`torch` (PyTorch)** performs the model's numerical calculations on a CPU or GPU.
- **`transformers`** is Hugging Face's library for loading and using pretrained models.
- **`AutoTokenizer`** loads Qwen's tokenizer: it converts text into numbered *tokens* (words or pieces of words) that the model can process, and converts generated tokens back into text.
- **`AutoModelForCausalLM`** loads the text-generation model. “Causal” means it predicts the next token from the preceding tokens.
- **`BitsAndBytesConfig`** describes how to compress the model's weights into 4-bit values to save GPU memory. It does not train the model.
- **`pipeline`** connects the tokenizer and model into a convenient text-generation tool. This Hugging Face helper is one component of our larger data-analysis workflow.

**Reading the setup:** `MODEL_ID` selects the model; `from_pretrained(...)` downloads its saved files on the first run and reuses the download cache later. `device` chooses NVIDIA GPU (`cuda`), Apple GPU (`mps`), or CPU. The numeric `dtype` and quantization options control how the model is stored and computed.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
USE_4BIT = True
device = "cuda" if torch.cuda.is_available() else (
    "mps" if torch.backends.mps.is_available() else "cpu"
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model_options = {"dtype": torch.float32 if device == "cpu" else torch.float16}
if device == "cuda":
    model_options["device_map"] = "auto"
    if USE_4BIT:
        model_options["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
        )
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **model_options)
if device != "cuda":
    model = model.to(device)
gen_pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

def ask_qwen(prompt, system="You are a helpful assistant.", max_new_tokens=512):
    messages = [{"role": "system", "content": system}, {"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return gen_pipe(text, max_new_tokens=max_new_tokens, do_sample=False,
                    return_full_text=False, pad_token_id=tokenizer.eos_token_id)[0]["generated_text"].strip()

print("Loaded", MODEL_ID, "on", device)

**How `ask_qwen()` works:**

1. Build two messages: `system` gives general instructions; `user` contains our question or task.
2. `apply_chat_template` adds the role markers and formatting Qwen expects. This helps it distinguish instructions from the user's message.
3. `gen_pipe` tokenizes the text, generates tokens, and converts them back to text. `max_new_tokens` limits the answer length; `do_sample=False` picks the most likely next token at each step.
4. `return_full_text=False` returns the new answer rather than repeating the input prompt. The helper extracts `generated_text` from the pipeline's result.

We can reuse this same helper for an explanation or for proposed code—the prompt determines the task.

## 2. Load one game and prepare our small table

This repeats the essential steps from Part 1. The cache path is relative to the notebook's working directory. To avoid another download, copy Part 1's JSON to that path. If the NHL API is unavailable, use the instructor's cached file.

**What the download cell does:**

- **`requests`** sends a GET request to the NHL API; **`Path`** handles the local cache filename and folders.
- If the file exists, we reuse it. Otherwise, `raise_for_status()` checks for HTTP errors, and the assertions check the game ID and event list before saving.
- **`json.dumps`** turns the downloaded Python object into JSON text for storage; **`json.loads`** reads that text back into a Python object. At the end, `game` is a dictionary containing the game's data.

In [ ]:
import json
from pathlib import Path
import requests

GAME_ID = 2025030311
raw_path = Path("data/raw") / f"{GAME_ID}.json"
print("Cached game:", raw_path.resolve())
if not raw_path.exists():
    response = requests.get(
        f"https://api-web.nhle.com/v1/gamecenter/{GAME_ID}/play-by-play", timeout=30,
    )
    response.raise_for_status()
    downloaded = response.json()
    assert downloaded["id"] == GAME_ID and isinstance(downloaded["plays"], list)
    raw_path.parent.mkdir(parents=True, exist_ok=True)
    raw_path.write_text(json.dumps(downloaded), encoding="utf-8")
game = json.loads(raw_path.read_text(encoding="utf-8"))
assert game["id"] == GAME_ID and isinstance(game["plays"], list)
print(game["awayTeam"]["abbrev"], "at", game["homeTeam"]["abbrev"], game["gameDate"])

### From nested JSON to a table

- **`pandas` (`pd`)** provides the DataFrame: a table with named columns. `pd.json_normalize(game["plays"])` flattens nested event fields into columns such as `details.shotType`.
- `.isin(...)` builds a filter for shots and goals; `.loc[...]` selects those rows and the columns we want. `.rename(...)` gives the columns shorter names, and `.copy()` makes a separate table to edit.
- `.map(team_names)` replaces numeric team IDs with abbreviations. `.eq("goal")` creates the Boolean `is_goal` column.
- We retain four columns, reset the row index, and check for duplicate event IDs and missing teams. **`display`** shows a preview in the notebook; **`Image`** will display the generated plot later.

In [ ]:
import pandas as pd
from IPython.display import display, Image

events = pd.json_normalize(game["plays"])
columns = {"eventId": "event_id", "typeDescKey": "event_type",
           "details.eventOwnerTeamId": "team_id", "details.shotType": "shot_type"}
shots = events.loc[events["typeDescKey"].isin(["shot-on-goal", "goal"]), list(columns)].rename(columns=columns).copy()
team_names = {game[side]["id"]: game[side]["abbrev"] for side in ["awayTeam", "homeTeam"]}
shots["team"] = shots["team_id"].map(team_names)
shots["is_goal"] = shots["event_type"].eq("goal")
shots = shots[["event_id", "team", "shot_type", "is_goal"]].reset_index(drop=True)
assert shots["event_id"].is_unique and shots["team"].notna().all()
display(shots.head())

**Define “shot” before asking the model.** Each row is a shot-on-goal event **or a goal**. Missed and blocked shots are excluded. Our goal percentage is `100 × goals / all rows`, grouped by team. Missing shot types remain missing and do not remove rows from team totals.

Compute reference answers first. For the classroom snapshot: MTL has **22 shots and 6 goals**; CAR has **28 shots and 2 goals**. These are checkpoints, not values to hard-code. This game is outside the milestone's requested seasons and cannot support season-wide conclusions.

**Reading the reference calculation:** `groupby("team")` separates the rows by shooting team. Inside `agg`, `size` counts all rows and `sum` adds the Boolean `is_goal` values (`True` counts as 1, `False` as 0). We then calculate the percentage from those two totals. `expected` is our reference table for checking the model's work.

In [ ]:
expected = shots.groupby("team").agg(shots=("event_id", "size"), goals=("is_goal", "sum"))
expected["goal_pct"] = 100 * expected["goals"] / expected["shots"]
display(expected)

## 3. Connect Qwen to PandasAI

This is the same adapter pattern as the draft. We explicitly tell the small model to use `dfs[0]` (the real table), rather than reconstructing data from the sample rows in its prompt. PandasAI constructs an analysis prompt; our `call()` method passes it through Qwen's chat template. The adapter contains no NHL analysis logic.

PandasAI 2 returns the answer directly from `.chat()`; inspect code on `sdf.last_code_executed`. Its code checks are **not a security sandbox**. Use the disposable runtime described above for these cells.

**What are these classes for?**

- **`SmartDataframe`** wraps our ordinary pandas table and adds `.chat()`. We still keep `shots` for manual analysis.
- **`LLM`** is PandasAI's base interface for a language model. `QwenPandasLLM(LLM)` implements that interface for our local Qwen model; it does not create or train a new model.
- **`call()`** receives PandasAI's instructions, converts them into text, and sends them to `ask_qwen()`. Its returned text is the proposed analysis code. `type` is just a label identifying this adapter.
- **`self.last_prompt` / `self.last_response`** keep the latest input and raw model output for inspection. `calls` counts how often Qwen is asked to generate text, including any internal repair requests.

**Connecting the pieces:** `pandas_llm` is an instance of our adapter; `config["llm"]` tells `SmartDataframe` to use it. `dfs[0]` is PandasAI's name for the first supplied DataFrame during code execution. `plot_dir` is where generated charts are saved, and `data_context` explains our columns and hockey definitions to the model.

`enable_cache=False` disables reuse of PandasAI answers. It does **not** disable the NHL JSON cache created earlier. These are two separate caches for two different steps.

In [ ]:
from pandasai import SmartDataframe
from pandasai.llm.base import LLM

class QwenPandasLLM(LLM):
    calls = 0

    @property
    def type(self):
        return "qwen-local"

    def call(self, instruction, context=None):
        self.calls += 1
        self.last_prompt = instruction.to_string() + (
            "\nUse the existing df = dfs[0]. Complete the calculation in the QUERY. "
            "Replace all TODOs and placeholder text with working Python. "
            "Do not read another file or recreate the sample data."
        )
        self.last_response = ask_qwen(
            self.last_prompt,
            system="You are a data analyst. Return executable pandas code in a fenced python block. "
                   "Follow the requested result dictionary format.",
            max_new_tokens=768,
        )
        return self.last_response

plot_dir = Path("data/plots").resolve()
plot_dir.mkdir(parents=True, exist_ok=True)
pandas_llm = QwenPandasLLM()
sdf = SmartDataframe(shots, config={
    "llm": pandas_llm, "enable_cache": False, "verbose": False,
    "save_logs": False, "max_retries": 1, "use_error_correction_framework": False,
    "save_charts": True, "save_charts_path": str(plot_dir), "open_charts": False,
})
data_context = (
    "This table contains only game 2025030311, MTL vs CAR on 2026-05-21. "
    "Each row is one shot INCLUDING goals. is_goal is Boolean. "
    "Use df = dfs[0] directly; it is already filtered to this game. "
    "The only columns are event_id, team, shot_type, is_goal. "
    "Missing shot_type does not exclude a row. "
)

## 4. Three questions, three checks

Start with a count, then an aggregation, then a chart. A returned error message or unusable result counts as a failed attempt. Answer caching is disabled. PandasAI may still make internal repair calls, so we also count model calls. If you revise a prompt, record the new attempt separately.

**First, ask for a number:**

- `data_context + ...` attaches our data definitions to the question, so the model knows what one row means.
- `sdf.chat(..., output_type="number")` asks PandasAI to generate and execute analysis code with a numeric result. Specifying the output type does not guarantee that the calculation is correct.
- **`Number`** lets us check that the answer is numeric. We then compare it with the reference value at `expected.loc["MTL", "goals"]` and store the outcome in `count_ok`. Printing the code helps explain a mismatch.

In [ ]:
from numbers import Number

count_prompt = data_context + "How many rows have team == 'MTL' and is_goal == True? Return a number."
count_answer = sdf.chat(count_prompt, output_type="number")
count_code = sdf.last_code_executed
print("Answer:", count_answer)
print("Generated code:\n", count_code)
count_ok = isinstance(count_answer, Number) and count_answer == int(expected.loc["MTL", "goals"])
print("Matches pandas:", count_ok)

### Ask for a table, then compare its contents

- The prompt specifies the output columns and grouping so we have a clear result to check.
- PandasAI 2 may wrap its answer in a `SmartDataframe`; `.dataframe` extracts the underlying pandas table.
- Before comparing, we use team names as the index, sort both tables, and select the same columns. This avoids treating a different team order as an error.
- **`pd.testing.assert_frame_equal`** checks the values against our reference. The options allow different numeric dtypes and tiny rounding differences. `try` / `except` records a failed comparison as `table_ok=False`, so we can inspect the problem and continue.

In [ ]:
table_prompt = data_context + (
    "Return a DataFrame with exactly three columns: team, shots (count of rows), "
    "goals (sum of is_goal). Group by team so there is one row per team."
)
table_answer = sdf.chat(table_prompt, output_type="dataframe")
table_code = sdf.last_code_executed
display(table_answer)
print(table_code)
# PandasAI 2 wraps tabular answers in a SmartDataframe.
actual = table_answer.dataframe if isinstance(table_answer, SmartDataframe) else table_answer
try:
    pd.testing.assert_frame_equal(
        actual.set_index("team").sort_index()[["shots", "goals"]], expected[["shots", "goals"]].sort_index(),
        check_dtype=False, check_exact=False, rtol=1e-6, atol=1e-6,
    )
    table_ok = True
except (AssertionError, AttributeError, KeyError, TypeError) as error:
    table_ok = False
    print("Mismatch to investigate:", error)
print("Matches pandas:", table_ok)

**Inspect the calculation:** does the generated code count all rows as shots, sum the Boolean `is_goal`, and group by team? Does it accidentally drop rows with missing shot type? A fluent explanation does not answer those questions.

**Next, request a plot:**

- `output_type="plot"` asks for plotting code. Python draws the chart and saves an image; Qwen generates the instructions, not the image pixels.
- The returned answer may be a PNG path. **`Image`** and **`display`** from `IPython.display` show that file inside the notebook; if no usable image is returned, we print the response instead.
- Inspect the generated code and compare the bar heights with `expected["goals"]`. A correct-looking title or attractive chart is not enough to validate the values.

In [ ]:
chart_prompt = data_context + (
    "Create a bar chart of total goals for each team: group by team and sum is_goal. "
    "Label the axes Team and Goals. Title: MTL vs CAR — one game. "
    "The output folder already exists; do not create directories."
)
chart_answer = sdf.chat(chart_prompt, output_type="plot")
chart_code = sdf.last_code_executed
if isinstance(chart_answer, str) and chart_answer.endswith(".png") and Path(chart_answer).is_file():
    display(Image(filename=chart_answer))
else:
    print(chart_answer)
print(chart_code)
display(expected[["goals"]])  # Compare these values with the bar heights.

## 5. Evaluate, then try an unsupported question

Mark the chart correct only if its values, labels, and two-team coverage agree with the reference. Three tasks are a classroom check, not a model benchmark. Keep the prompts and generated code, including failed attempts.

If Qwen produces no usable chart, the manual reference `expected["goals"].plot.bar(xlabel="Team", ylabel="Goals")` lets you continue the discussion. Label it as manual and mark the **LLM chart attempt** as failed. In our local preparation, counting succeeded, while table-format and plotting errors still occurred; the tutorial does not assume three successful answers.

**Reading the evaluation cell:** `count_ok` and `table_ok` come from automatic comparisons; `chart_ok` starts as `None` because it needs your visual review. Once all three have been marked, converting the Booleans to numbers and taking their mean gives the fraction of successful tasks. The model-call counter is separate: one task may require more than one generation.

In [ ]:
chart_ok = None  # Replace with True or False after inspecting the chart; an error is False.
evaluation = pd.DataFrame({
    "task": ["MTL goal count", "Team summary", "Goals-by-team chart"],
    "correct": [count_ok, table_ok, chart_ok],
})
display(evaluation)
print("Model calls, including internal repairs:", pandas_llm.calls)
if evaluation["correct"].notna().all():
    print(f"Success rate: {evaluation['correct'].astype(bool).mean():.0%}")
else:
    print("Review the chart before reporting a success rate.")

**Troubleshooting:** if code is truncated, increase `max_new_tokens` in the adapter. If extraction or execution fails, inspect the prompt/code and simplify the question; do not assume the small model can solve every task. CPU generation can take substantially longer than GPU generation.

**Sources:** [Qwen model card](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct) · [PandasAI 2.3.2](https://pypi.org/project/pandasai/2.3.2/). This notebook intentionally follows the draft's **v2 API**, not v3 examples found elsewhere.